# Analiza streszczeń literatury młodzieżowej

## Import bibliotek

In [ ]:
import os
import json
import string
import morfeusz2
import pandas as pd
import numpy as np
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import  dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
import nltk
if "." not in nltk.data.path:
    nltk.data.path.append(".")
    nltk.download("punkt_tab")
        


In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [ ]:
 def create_dir(path):
        if not os.path.exists(path):
            os.mkdir(path)

## utworzenie korpusu dokumnetów 

In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()
files_names 

In [ ]:
documents_dict = {}
for file_name in files_names:
     documents_dict[file_name] = corpus.raw(file_name)
print_dict(documents_dict)

In [ ]:
documnets_lengths = {}
for file_name in files_names:
     documnets_lengths[file_name] = {
"pre": len(word_tokenize(documents_dict[file_name], language="polish"))
}
print_dict(documnets_lengths)

## Utworzenie macierzy częstości przed wstępnym przygotowaniem dokumentów 

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df)

In [ ]:
cv = CountVectorizer()
tv = TfidfVectorizer()

In [ ]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [ ]:
matrices_parameters = {
    "pre" : {
        "words" : matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}
print_dict(matrices_parameters)

## Wstępne przygotowanie dokumentów do analizy

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="UTF-8")
stopwords = stopwords_file.read().splitlines()
stopwords_file.close()
stopwords = stopwords[4:]
print(stopwords, end=", ")

In [ ]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text),op.itemgetter(0,1))
    def disambiguate(group):
        pairs =((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        _, lemma = min(pairs)
        return lemma.split(":")[0]
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))


In [ ]:
morfer = morfeusz2.Morfeusz()
morfer.analyse("Ala ma kota. Kot lubi mleko")

In [ ]:
lemmatize("Ala ma kota")


In [ ]:
for key in documents_dict:
 documents_dict[key] = documents_dict[key].lower()
 documents_dict[key] = "".join([char for char in documents_dict[key] if char not in string.punctuation])
 documents_dict[key] = lemmatize(documents_dict[key])
 documents_dict[key] = " ".join([word for word in word_tokenize(documents_dict[key], language="polish") if word not in stopwords])
print_dict(documents_dict)